# EDA & Préparation ML — ObRail Europe
## Substitution Avion → Train

**Enjeu :** Identifier automatiquement les lignes candidates à la substitution avion par train  
**Source :** `donnee/staging_fact_route_analysis.csv` — fichier enrichi complet  
**CO2 :** Méthodologie EcoPassenger (UIC/IFEU 2016)  
**Label :** `is_substitutable = 1` si distance ≤ 600 km ET vol existant (loi française 2023)  

Ce notebook couvre :
- Section 1 : Analyse Exploratoire (EDA)
- Section 2 : Préparation des données ML

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import os
import warnings
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')
os.makedirs('../docs', exist_ok=True)
os.makedirs('../data', exist_ok=True)

df = pd.read_csv('../donnee/staging_fact_route_analysis.csv')
print(f'Dataset chargé : {len(df)} corridors, {len(df.columns)} colonnes')
print(f'Colonnes : {list(df.columns)}')

---
# SECTION 1 — Analyse Exploratoire (EDA)

## 1.1 Vue d'ensemble du dataset

In [ ]:
info = pd.DataFrame({
    'Type': df.dtypes,
    'Non-null': df.count(),
    'Null': df.isnull().sum(),
    'Null %': (df.isnull().sum() / len(df) * 100).round(1)
})
print(info.to_string())

In [ ]:
num_cols = ['distance_km','co2_train_kg','co2_avion_kg','co2_saved_kg',
            'origin_station_traffic','origin_city_population',
            'dest_station_traffic','dest_city_population']
df[num_cols].describe().round(2)

## 1.2 Distribution de la cible : `is_substitutable`

In [ ]:
counts = df['is_substitutable'].value_counts()
labels = ['Non substituable (0)', 'Substituable (1)']
colors = ['#e74c3c', '#2ecc71']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(labels, [counts[0], counts[1]], color=colors, edgecolor='white')
axes[0].set_title('Distribution is_substitutable')
axes[0].set_ylabel('Nombre de corridors')
for i, v in enumerate([counts[0], counts[1]]):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

axes[1].pie([counts[0], counts[1]], labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Répartition (%)')

plt.suptitle(f'Déséquilibre : {counts[1]/len(df)*100:.1f}% substituables / {counts[0]/len(df)*100:.1f}% non-substituables')
plt.tight_layout()
plt.savefig('../docs/fig_is_substitutable.png', dpi=150, bbox_inches='tight')
plt.show()
print('⚠️  Déséquilibre → utiliser class_weight="balanced" dans les modèles')

## 1.3 Distributions des features CO2 et distance

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
features = [
    ('distance_km',  'Distance (km)',      '#3498db'),
    ('co2_train_kg', 'CO2 Train (kg)',     '#27ae60'),
    ('co2_avion_kg', 'CO2 Avion (kg)',     '#e74c3c'),
    ('co2_saved_kg', 'CO2 Économisé (kg)', '#9b59b6'),
]
for ax, (col, label, color) in zip(axes.flatten(), features):
    data = df[col].dropna()
    ax.hist(data, bins=40, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(data.mean(),   color='black',  linestyle='--', linewidth=1.5, label=f'Moy: {data.mean():.1f}')
    ax.axvline(data.median(), color='orange', linestyle='--', linewidth=1.5, label=f'Méd: {data.median():.1f}')
    ax.set_title(label)
    ax.legend(fontsize=9)

plt.suptitle('Distribution des features CO2 et distance', fontsize=14)
plt.tight_layout()
plt.savefig('../docs/fig_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.4 Distribution fréquentation et population

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fréquentation gares (origin uniquement pour éviter doublon)
traffic = df['origin_station_traffic'].dropna()
axes[0].hist(traffic / 1e6, bins=30, color='#1abc9c', alpha=0.8, edgecolor='white')
axes[0].set_title(f'Fréquentation gare origine (n={len(traffic)})')
axes[0].set_xlabel('Millions de voyageurs/an')
axes[0].set_ylabel('Nombre de corridors')
axes[0].axvline(traffic.median()/1e6, color='orange', linestyle='--', label=f'Méd: {traffic.median()/1e6:.1f}M')
axes[0].legend()

# Population ville
pop = df['origin_city_population'].dropna()
axes[1].hist(pop / 1e3, bins=30, color='#e67e22', alpha=0.8, edgecolor='white')
axes[1].set_title(f'Population ville origine (n={len(pop)})')
axes[1].set_xlabel('Population (milliers)')
axes[1].axvline(pop.median()/1e3, color='black', linestyle='--', label=f'Méd: {pop.median()/1e3:.0f}k')
axes[1].legend()

plt.suptitle('Features démographiques', fontsize=13)
plt.tight_layout()
plt.savefig('../docs/fig_demographic.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Couverture fréquentation : {len(traffic)}/{len(df)} ({len(traffic)/len(df)*100:.1f}%)')
print(f'Couverture population    : {len(pop)}/{len(df)} ({len(pop)/len(df)*100:.1f}%)')

## 1.5 CO2 économisé vs Distance (par classe)

In [ ]:
df_both = df.dropna(subset=['co2_avion_kg'])
colors_map = {1: '#2ecc71', 0: '#e74c3c'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for val, grp in df_both.groupby('is_substitutable'):
    label = 'Substituable' if val == 1 else 'Non substituable'
    axes[0].scatter(grp['distance_km'], grp['co2_saved_kg'],
                    c=colors_map[val], label=label, alpha=0.5, s=15)
axes[0].axvline(600, color='black', linestyle='--', linewidth=1.5, label='Seuil 600km')
axes[0].set_xlabel('Distance (km)')
axes[0].set_ylabel('CO2 économisé (kg)')
axes[0].set_title('CO2 économisé vs Distance')
axes[0].legend()

order = df_both.groupby('vehicule_type')['co2_saved_kg'].median().sort_values(ascending=False).index
sns.boxplot(data=df_both, x='vehicule_type', y='co2_saved_kg', order=order, ax=axes[1], palette='Set2')
axes[1].set_title('CO2 économisé par type de train')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../docs/fig_co2_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Gain CO2 moyen : {df_both["co2_saved_kg"].mean():.1f} kg/passager')
print(f'Facteur CO2 avion/train (médiane) : x{(df_both["co2_avion_kg"]/df_both["co2_train_kg"]).median():.0f}')

## 1.6 Matrice de corrélations

In [ ]:
corr_cols = ['distance_km','co2_train_kg','co2_avion_kg','co2_saved_kg',
             'origin_station_traffic','origin_city_population','is_substitutable']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            mask=mask, ax=ax, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Matrice de corrélations')
plt.tight_layout()
plt.savefig('../docs/fig_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print('Corrélations avec is_substitutable :')
print(corr['is_substitutable'].drop('is_substitutable').sort_values().round(3))

## 1.7 Valeurs manquantes

In [ ]:
nulls = df.isnull().sum()
null_df = pd.DataFrame({'Nulls': nulls, '%': (nulls/len(df)*100).round(1)}).query('Nulls > 0')
print(null_df.to_string())
print()
print('→ co2_avion_kg / co2_saved_kg : 132 corridors sans vol → is_substitutable=0, exclure du Modèle 2')
print('→ *_station_traffic : gares non-françaises sans données SNCF → remplacer par 0')
print('→ *_city_population : villes non-reconnues → remplacer par 0')

## 1.8 Visualisation géographique des corridors

In [ ]:
m = folium.Map(location=[46.5, 2.5], zoom_start=5, tiles='CartoDB positron')
df_geo = df.dropna(subset=['station_lat','station_long','station_lat_dest','station_long_dest'])

for _, row in df_geo.iterrows():
    color = '#2ecc71' if row['is_substitutable'] == 1 else '#e74c3c'
    co2_txt = f"{row['co2_saved_kg']:.1f}kg" if pd.notna(row.get('co2_saved_kg')) else 'N/A'
    folium.PolyLine(
        locations=[[row['station_lat'], row['station_long']],
                   [row['station_lat_dest'], row['station_long_dest']]],
        color=color, weight=1.5, opacity=0.5,
        tooltip=f"{row['origin']} → {row['destination']} ({row['distance_km']:.0f}km, CO2 saved: {co2_txt})"
    ).add_to(m)

m.save('../docs/carte_corridors.html')
print(f'Carte sauvegardée : docs/carte_corridors.html ({len(df_geo)} corridors tracés)')
display(m)

## 1.9 Tableau des variables retenues (livrable sujet)

In [ ]:
variables = pd.DataFrame([
    {'Variable':'distance_km',            'Rôle':'Feature',          'Justification':'Corrélation -0.40 avec cible | détermine si < 600km'},
    {'Variable':'co2_train_kg',           'Rôle':'Feature',          'Justification':'Varie selon mix électrique pays et type de train'},
    {'Variable':'co2_avion_kg',           'Rôle':'Feature',          'Justification':'Présence = vol existant. NULL → is_substitutable=0'},
    {'Variable':'co2_saved_kg',           'Rôle':'Cible Modèle 2',   'Justification':'co2_avion - co2_train | aucun biais circulaire'},
    {'Variable':'vehicule_type',          'Rôle':'Feature (encodée)','Justification':'TGV vs Train Nuit = profils CO2 très différents'},
    {'Variable':'origin_station_traffic', 'Rôle':'Feature',          'Justification':'Fréquentation SNCF | proxy demande. NULL → 0'},
    {'Variable':'origin_city_population', 'Rôle':'Feature',          'Justification':'Population INSEE/GeoNames | densité demande. NULL → 0'},
    {'Variable':'dest_station_traffic',   'Rôle':'Feature',          'Justification':'Idem côté destination'},
    {'Variable':'dest_city_population',   'Rôle':'Feature',          'Justification':'Idem côté destination'},
    {'Variable':'is_substitutable',       'Rôle':'Cible Modèle 1',   'Justification':'Loi FR 2023 : distance ≤ 600km ET vol existant'},
    {'Variable':'station_lat/lon',        'Rôle':'Carte uniquement', 'Justification':'Coordonnées GPS pour visualisation Folium'},
    {'Variable':'traffic_share_pct',      'Rôle':'❌ Supprimée',     'Justification':'100% NULL | aucune donnée disponible'},
])
pd.set_option('display.max_colwidth', 60)
variables

---
# SECTION 2 — Préparation des données ML

## 2.1 Sélection et nettoyage des features

In [ ]:
# Features retenues pour les deux modèles
FEATURES = [
    'distance_km',
    'co2_train_kg',
    'co2_avion_kg',
    'vehicule_type',          # → encodage
    'origin_station_traffic',
    'origin_city_population',
    'dest_station_traffic',
    'dest_city_population',
]
TARGET_M1 = 'is_substitutable'
TARGET_M2 = 'co2_saved_kg'

df_ml = df[FEATURES + [TARGET_M1, TARGET_M2]].copy()

# Remplacement des NULL par 0 pour trafic et population
for col in ['origin_station_traffic','origin_city_population','dest_station_traffic','dest_city_population']:
    df_ml[col] = df_ml[col].fillna(0)

# co2_avion_kg : NULL = pas de vol → remplacer par 0 pour le Modèle 1
df_ml['co2_avion_kg'] = df_ml['co2_avion_kg'].fillna(0)

print(f'Dataset ML : {len(df_ml)} lignes, {len(df_ml.columns)} colonnes')
print(f'Valeurs manquantes restantes : {df_ml.isnull().sum().sum()}')

## 2.2 Encodage de `vehicule_type`

In [ ]:
le = LabelEncoder()
df_ml['vehicule_type_enc'] = le.fit_transform(df_ml['vehicule_type'])

print('Mapping vehicule_type :')
for i, cls in enumerate(le.classes_):
    print(f'  {i} → {cls}')

# On remplace la colonne texte par l'encodage
df_ml = df_ml.drop(columns=['vehicule_type'])
df_ml = df_ml.rename(columns={'vehicule_type_enc': 'vehicule_type'})

## 2.3 Dataset Modèle 1 — Classification `is_substitutable`

In [ ]:
FEATURES_M1 = [c for c in FEATURES if c != 'vehicule_type'] + ['vehicule_type']
FEATURES_M1 = [c if c != 'vehicule_type' else 'vehicule_type' for c in df_ml.columns if c not in [TARGET_M1, TARGET_M2]]

X_m1 = df_ml[FEATURES_M1]
y_m1 = df_ml[TARGET_M1]

X_train_m1, X_temp, y_train_m1, y_temp = train_test_split(
    X_m1, y_m1, test_size=0.30, random_state=42, stratify=y_m1
)
X_val_m1, X_test_m1, y_val_m1, y_test_m1 = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print('=== Modèle 1 — Classification ===')
print(f'Train : {len(X_train_m1)} lignes | Val : {len(X_val_m1)} | Test : {len(X_test_m1)}')
print(f'Train is_sub=1 : {y_train_m1.mean()*100:.1f}% | Test is_sub=1 : {y_test_m1.mean()*100:.1f}%')

## 2.4 Dataset Modèle 2 — Régression `co2_saved_kg`

In [ ]:
# Pour la régression : on garde uniquement les corridors avec un vol existant
df_m2 = df_ml[df[TARGET_M2].notna()].copy()
df_m2[TARGET_M2] = df[TARGET_M2][df[TARGET_M2].notna()].values

X_m2 = df_m2[FEATURES_M1]
y_m2 = df_m2[TARGET_M2]

X_train_m2, X_temp, y_train_m2, y_temp = train_test_split(
    X_m2, y_m2, test_size=0.30, random_state=42
)
X_val_m2, X_test_m2, y_val_m2, y_test_m2 = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print('=== Modèle 2 — Régression ===')
print(f'Lignes utilisables (avec vol) : {len(df_m2)}')
print(f'Train : {len(X_train_m2)} | Val : {len(X_val_m2)} | Test : {len(X_test_m2)}')
print(f'co2_saved_kg — moy: {y_m2.mean():.1f} | méd: {y_m2.median():.1f} | std: {y_m2.std():.1f}')

## 2.5 Normalisation des features numériques

In [ ]:
# StandardScaler : moyenne=0, écart-type=1
# On fit sur le train uniquement, on applique sur val et test
scaler = StandardScaler()

X_train_m1_scaled = scaler.fit_transform(X_train_m1)
X_val_m1_scaled   = scaler.transform(X_val_m1)
X_test_m1_scaled  = scaler.transform(X_test_m1)

print('Normalisation appliquée (StandardScaler)')
print(f'Moyenne train avant : {X_train_m1["distance_km"].mean():.1f}km')
print(f'Moyenne train après : {X_train_m1_scaled[:,0].mean():.4f} (≈0)')
print()
print('Scaler sauvegardé pour usage en production')

import joblib
os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler.joblib')
joblib.dump(le,     '../models/label_encoder_vehicule.joblib')
print('Sauvegardé : models/scaler.joblib + models/label_encoder_vehicule.joblib')

## 2.6 Sauvegarde des datasets ML

In [ ]:
# Sauvegarde en CSV pour réutilisation dans les scripts d'entraînement
X_train_m1.assign(is_substitutable=y_train_m1.values).to_csv('../data/train_m1.csv', index=False)
X_val_m1.assign(is_substitutable=y_val_m1.values).to_csv('../data/val_m1.csv', index=False)
X_test_m1.assign(is_substitutable=y_test_m1.values).to_csv('../data/test_m1.csv', index=False)

X_train_m2.assign(co2_saved_kg=y_train_m2.values).to_csv('../data/train_m2.csv', index=False)
X_val_m2.assign(co2_saved_kg=y_val_m2.values).to_csv('../data/val_m2.csv', index=False)
X_test_m2.assign(co2_saved_kg=y_test_m2.values).to_csv('../data/test_m2.csv', index=False)

print('Datasets sauvegardés dans data/ :')
print('  Modèle 1 : train_m1.csv | val_m1.csv | test_m1.csv')
print('  Modèle 2 : train_m2.csv | val_m2.csv | test_m2.csv')

---
## Synthèse EDA + Préparation

| Point clé | Valeur | Impact ML |
|---|---|---|
| Dataset complet | 1509 corridors, 18 colonnes | Source unique |
| Déséquilibre classes | 85.8% / 14.2% | `class_weight='balanced'` |
| Feature la plus corrélée | `co2_avion_kg` (-0.77) | Feature principale M1 |
| Gain CO2 moyen | ~90 kg/passager | Signal fort M2 |
| Couverture fréquentation | ~46% | NULL → 0 |
| Couverture population | ~68% | NULL → 0 |
| Split train/val/test | 70/15/15 (stratifié M1) | Reproductible (seed=42) |
| Scaler + LabelEncoder | Sauvegardés en .joblib | Réutilisables en prod |